I have the Titanic dataset loaded as titanic.csv in the current
directory.
Please write a Python script that:
1. Loads titanic.csv using pandas
2. Prints a schema summary: column names, dtypes, and count of
missing values per column
3. Cleans the dataset as follows:
- Drop columns that are identifiers or too granular to be
useful: PassengerId, Name, Ticket, Cabin
- Impute missing Age values with the median age
- Impute missing Embarked values with the mode
- Encode the Sex column as binary (male=0, female=1)
4. Prints a clear summary of every transformation applied (e.g.
"Imputed 177 missing Age values with median: 28.0")
5. Prints the shape of the final clean dataframe
Use RANDOM_STATE = 42 where any randomness is involved. Do not
drop any passenger rows.
Save the cleaned dataframe as titanic_clean.csv.

Comments: completed in 5s

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('titanic.csv')

# Print schema summary
print("Schema Summary:")
for col in df.columns:
    print(f"{col}: {df[col].dtype}, missing: {df[col].isnull().sum()}")

# Clean the dataset
# Drop specified columns
dropped_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df.drop(dropped_cols, axis=1, inplace=True)
print(f"Dropped columns: {', '.join(dropped_cols)}")

# Impute missing Age with median
missing_age = df['Age'].isnull().sum()
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)
print(f"Imputed {missing_age} missing Age values with median: {median_age}")

# Impute missing Embarked with mode
missing_embarked = df['Embarked'].isnull().sum()
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'].fillna(mode_embarked, inplace=True)
print(f"Imputed {missing_embarked} missing Embarked values with mode: '{mode_embarked}'")

# Encode Sex as binary
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
print("Encoded Sex column as binary: male=0, female=1")

# Print final shape
print(f"Final dataframe shape: {df.shape}")

# Save cleaned dataframe
df.to_csv('titanic_clean.csv', index=False)

Using the cleaned Titanic dataset (titanic_clean.csv, with
Survived as the target column),
please write a Python script that:
1. Loads titanic_clean.csv
2. Splits the data into features (X) and target (y = Survived)
3. Performs a stratified 80/20 train/test split using
RANDOM_STATE = 42
4. Trains a Logistic Regression classifier on the training set
only
(use max_iter=1000, random_state=42)
5. Evaluates the model on the test set and prints:
- Accuracy
- Precision (weighted)
- Recall (weighted)
- F1 score (weighted)
6. Prints a confusion matrix
Important constraints:
- The model must only be fit on training data, never on test data
- All preprocessing must happen after the train/test split, or
use a Pipeline
- Use RANDOM_STATE = 42 throughout

Completed in 4s

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load the cleaned dataset
df = pd.read_csv('titanic_clean.csv')

# Split into features (X) and target (y)
X = df.drop('Survived', axis=1)
y = df['Survived']

# Perform stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train Logistic Regression classifier on training set
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluate on test set
y_pred = model.predict(X_test)

# Print metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (weighted): {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall (weighted): {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

I have a Python pipeline script called titanic_broken_leakage.py.
I suspect it contains a data leakage bug.
Please:
1. Read the script carefully
2. Identify any data leakage or evaluation errors and explain
precisely why they are a problem
3. Fix the bug(s)
4. Produce a corrected version of the script called
titanic_fixed_leakage.py
5. Run both the original and the corrected version and report the
metrics from each,
so I can see whether the leakage was inflating performance

Complete in 12s, 3 re-runs. Below Copilots answer

### 1) Data leakage / evaluation error found

In titanic_leakage.py, the scaler is fitted **before** splitting into train/test:

- `scaler.fit_transform(X)` is called on the full dataset.
- Then `train_test_split` is applied.

This leaks information from the test set into the training process, because the feature scaling (mean/variance) is computed using **all** data, including the test portion. That can inflate performance metrics (because the model indirectly “sees” test distribution during preprocessing).

### 2) Fix

Split the data first, then fit the scaler only on `X_train` and apply it to both splits.

---

### 3) Corrected version: `titanic_fixed_leakage.py`



In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv("titanic_clean.csv")
X = df.drop("Survived", axis=1)
y = df["Survived"]

# Split before scaling to avoid leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1: {f1_score(y_test, y_pred, average='weighted'):.4f}")



---

### 4) Running both scripts & reporting metrics

Run these in the terminal from titanic-benchmark:



In [ ]:
python titanic_leakage.py
python titanic_fixed_leakage.py



The first run (leaky version) will typically show *higher* metrics than the fixed version, because the scaler “cheated” by using test-set statistics.

> I can’t execute them here, but running the commands above in your environment will give you the exact metric values for each script so you can see how much performance was inflated by leakage.

 I have a Python script called titanic_broken_debug.py that is
supposed to train a
Logistic Regression model on the Titanic dataset and evaluate it
properly.
However, I believe it contains at least two bugs that affect
either correctness or
evaluation validity.
Please:
1. Read the script carefully and identify all bugs
2. Explain each bug and why it is a problem
3. Produce a corrected version called titanic_fixed_debug.py
4. Print the metrics from the corrected version
Do not run the original broken script — just analyse it and fix
it.

Complete in 12s. Below the answer of copilot

1) **Bugs found**

- **Wrong predictions used for evaluation**  
  The script computes `y_pred = model.predict(X_train)` but then compares it to `y_test`. That mismatches train vs test sets, so the reported metrics are meaningless.

- **Train/test split is not randomized**  
  `train_test_split(..., shuffle=False)` means the last 20% of the dataset becomes the test set. If the CSV is sorted (e.g., by passenger id, class, etc.), the test set isn’t a representative sample. Also, `random_state` has no effect when `shuffle=False`.

2) **Why they are problems**

- Using `X_train` predictions against `y_test` makes the evaluation invalid (you’re not measuring generalization).
- Non-shuffled split can cause overlap in distributions between train/test or make the test set unrepresentative, giving misleading metrics.

3) **Corrected version (`titanic_fixed_debug.py`)**



In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

df = pd.read_csv('titanic_clean.csv')

X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=RANDOM_STATE
)

model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1: {f1_score(y_test, y_pred, average='weighted'):.4f}")



4) **Metrics output (when running the corrected script)**

When you run `python titanic_fixed_debug.py`, it will print something like:



In [ ]:
Accuracy: 0.XXX0
Precision: 0.XXX0
Recall: 0.XXX0
F1: 0.XXX0



(Exact values depend on the dataset contents.)

I have the cleaned Titanic dataset as `titanic_clean.csv` in the current directory.

Please write a Python script that performs exploratory data analysis (EDA) on this dataset.

The script must:
1. Load `titanic_clean.csv` and print: shape, column names and dtypes, and the output of `.describe()`.
2. Compute and print survival rates broken down by:
   - Sex (male vs female)
   - Pclass (1st, 2nd, 3rd)
   Print the values as percentages rounded to 1 decimal place.
3. Generate and save the following four plots as PNG files in the current directory:
   a. `eda_survival_count.png` — bar chart of Survived (0 vs 1) counts
   b. `eda_age_distribution.png` — histogram of Age with 20 bins, coloured by Survived
   c. `eda_survival_by_sex.png` — grouped bar chart of survival rate by Sex
   d. `eda_correlation_heatmap.png` — correlation heatmap of all numeric features
4. Use `matplotlib.use('Agg')` at the top so plots save without needing a display.
5. Print a confirmation message for each plot saved (e.g. `"Saved eda_survival_count.png"`).

Use `RANDOM_STATE = 42`. Do not display plots interactively. Only use `titanic_clean.csv` as input; do not re-download or re-clean the data. You are not allowed to access any other scripts/documents.

Comments: completed in 5s

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

# Load the dataset
df = pd.read_csv('titanic_clean.csv')

# Print shape, column names and dtypes, and describe
print("Shape:", df.shape)
print("\nColumn names and dtypes:")
print(df.dtypes)
print("\nDescribe:")
print(df.describe())

# Compute and print survival rates
print("\nSurvival rates by Sex:")
survival_by_sex = df.groupby('Sex')['Survived'].mean() * 100
print(f"Male: {survival_by_sex[0]:.1f}%")
print(f"Female: {survival_by_sex[1]:.1f}%")

print("\nSurvival rates by Pclass:")
survival_by_pclass = df.groupby('Pclass')['Survived'].mean() * 100
for pclass in sorted(survival_by_pclass.index):
    print(f"Class {pclass}: {survival_by_pclass[pclass]:.1f}%")

# Generate and save plots
# a. Bar chart of Survived counts
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Survived')
plt.title('Survival Count')
plt.savefig('eda_survival_count.png')
print("Saved eda_survival_count.png")

# b. Histogram of Age with 20 bins, coloured by Survived
plt.figure(figsize=(8, 6))
sns.histplot(data=df, x='Age', bins=20, hue='Survived', multiple='stack')
plt.title('Age Distribution by Survival')
plt.savefig('eda_age_distribution.png')
print("Saved eda_age_distribution.png")

# c. Grouped bar chart of survival rate by Sex
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x='Sex', y='Survived', estimator=lambda x: sum(x)/len(x)*100)
plt.ylabel('Survival Rate (%)')
plt.title('Survival Rate by Sex')
plt.savefig('eda_survival_by_sex.png')
print("Saved eda_survival_by_sex.png")

# d. Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.savefig('eda_correlation_heatmap.png')
print("Saved eda_correlation_heatmap.png")
